# Run 10 — ResNet-50 (ImageNet pretrained) + CBAM, image-only, cropped

Single-stream Kaggle T4×2 recipe. No pose, no MediaPipe, no EMA, no CutMix.
Strips the pose stream from Run 9 to isolate the contribution of the deeper
pretrained backbone. Inputs are cropped to the left 80% of the frame so the
model stops attending to the empty area behind the driver seat.

Defaults (different from Run 6/8/9):
- `--img-size 320`
- ImageNet normalization (no `--dataset-stats`) — required for `--pretrained`.
- `--crop-left-frac 0.8` (applied inside `StateFarmCropDataset`).
- No EMA, no CutMix — raw cross-entropy with label smoothing only.

Pre-reqs: `splits/train.csv`, `splits/val.csv` from prior runs. `stats.json`
is **not** needed unless you pass `--dataset-stats`.

## 1a. Paths

In [ ]:
import os, sys
COMP_DIR = "/kaggle/input/competitions/state-farm-distracted-driver-detection"
CODE_DIR = "/kaggle/input/driver-distraction-cbam"   # or /kaggle/working/code
WORK     = "/kaggle/working"
RUN      = f"{WORK}/run10"

assert os.path.exists(COMP_DIR + "/driver_imgs_list.csv"), "Competition dataset not attached"
assert os.path.exists(CODE_DIR + "/train_run10.py"),       "Run 10 code missing (train_run10.py)"
sys.path.insert(0, CODE_DIR)
print("OK. GPU count:", __import__("torch").cuda.device_count())

### If using the GitHub mirror (skip if dataset attached):
```python
!rm -rf /kaggle/working/code
!git clone https://github.com/nxtruoong/DoAnCS231-V2 /kaggle/working/code
CODE_DIR = "/kaggle/working/code"
sys.path.insert(0, CODE_DIR)
```

## 1b. Splits prep — only if `splits/train.csv` missing

Run 10 uses ImageNet stats by default, so `stats.json` is optional. If you
still want it (for `--dataset-stats` ablation), keep `data_prep.py` in this
step.

In [ ]:
import subprocess, os
if not os.path.exists(f"{WORK}/splits/train.csv"):
    subprocess.run([
        "python", f"{CODE_DIR}/data_prep.py",
        "--data-root", COMP_DIR,
        "--out-dir",   f"{WORK}/splits",
        "--batch-size", "64",
        "--num-workers", "4",
    ], check=True)
else:
    print("splits/train.csv already exists, skipping")

## 2. Smoke test — 2 epochs (~5–7 min)

Verifies the pretrained-load path + ImageNet stats + crop pipeline.

In [ ]:
import subprocess
subprocess.run([
    "python", f"{CODE_DIR}/train_run10.py",
    "--data-root", COMP_DIR,
    "--splits-dir", f"{WORK}/splits",
    "--out-dir",    f"{WORK}/run10_smoke",
    "--backbone", "resnet50",
    "--pretrained",
    "--epochs", "2",
    "--batch-size", "32",
    "--num-workers", "4",
    "--lr", "0.01",
    "--warmup-epochs", "1",
    "--img-size", "320",
    "--crop-left-frac", "0.8",
    "--data-parallel",
], check=True)

**Expect:** 2 epochs complete, no OOM. ImageNet init + crop → val acc 0.45–0.75
by ep 2. If OOM → drop `--batch-size 24` or `--img-size 288`.

## 3. Full Run 10 training (~2–2.5 hr)

In [ ]:
import subprocess
subprocess.run([
    "python", f"{CODE_DIR}/train_run10.py",
    "--data-root", COMP_DIR,
    "--splits-dir", f"{WORK}/splits",
    "--out-dir",    RUN,
    "--backbone", "resnet50",
    "--pretrained",
    "--epochs", "30",
    "--batch-size", "32",
    "--num-workers", "4",
    "--lr", "0.01",
    "--warmup-epochs", "2",
    "--weight-decay", "1e-4",
    "--label-smoothing", "0.1",
    "--img-size", "320",
    "--crop-left-frac", "0.8",
    "--early-stop-patience", "8",
    "--early-stop-min-delta", "0.005",
    "--ckpt-every", "5",
    "--data-parallel",
], check=True)

**Milestones (target val acc — no EMA column):**

| ep | target val acc | Run 6 actual |
|---:|---:|---:|
| 05 | ≥ 0.70 | ~0.50 |
| 10 | ≥ 0.82 | ~0.78 |
| 20 | ≥ 0.87 | 0.82  |
| 30 | ≥ 0.88 | —     |

If val acc stalls below 0.80 by ep 10 → `--lr` too high; halve to 0.005.

## 4. Peek at history

In [ ]:
import json
hist = json.loads(open(f"{RUN}/history.json").read())
print(f"Last epoch: {hist[-1]['epoch']}")
best_idx = max(range(len(hist)), key=lambda i: hist[i]['val_acc'])
print(f"Best val: {hist[best_idx]['val_acc']:.4f} at ep {hist[best_idx]['epoch']}")

## 5. Eval + figures (~3 min)

`eval_run10.py` reads `backbone`, `pretrained`, `dataset_stats`,
`crop_left_frac` from the ckpt args. No extra flags needed.

In [ ]:
import subprocess
subprocess.run([
    "python", f"{CODE_DIR}/eval_run10.py",
    "--ckpt",        f"{RUN}/best.pt",
    "--data-root",   COMP_DIR,
    "--splits-dir",  f"{WORK}/splits",
    "--out-dir",     f"{RUN}/eval",
    "--history-json", f"{RUN}/history.json",
    "--batch-size", "64",
    "--num-workers", "4",
    "--img-size", "320",
], check=True)

## 6. Compare to Run 6 / 8 / 9

In [ ]:
import json, pandas as pd, os

def metrics(p):
    m = json.load(open(p))
    return {"accuracy": m["accuracy"],
            "macro_f1": m["macro avg"]["f1-score"],
            "weighted_f1": m["weighted avg"]["f1-score"]}

rows = {}
for label, path in [
    ("Run 6 (R18 scratch + CBAM)",   f"{WORK}/run6/eval/metrics.json"),
    ("Run 8 (R18 + pose)",            f"{WORK}/run8/eval/metrics.json"),
    ("Run 9 (R50 ImageNet + pose)",   f"{WORK}/run9/eval/metrics.json"),
    ("Run 10 (R50 ImageNet, cropped)", f"{RUN}/eval/metrics.json"),
]:
    if os.path.exists(path):
        rows[label] = metrics(path)
table = pd.DataFrame(rows).T
print(table.to_string())
table.to_csv(f"{WORK}/run10_vs_others.csv")

**Pass criterion for Run 10:** macro F1 ≥ 0.88, beating Run 6 headline 0.873.
If Run 10 macro F1 < Run 6 → ImageNet+depth alone don't help on this cabin
imagery; report Run 9 (pose fusion) as headline.

## 7. Bundle artifacts for download

In [ ]:
import zipfile
from pathlib import Path

OUT = Path(f"{WORK}/artifacts_run10.zip")
OUT.unlink(missing_ok=True)

paths = [
    Path(f"{RUN}/best.pt"),
    Path(f"{RUN}/history.json"),
    *Path(f"{RUN}/eval").iterdir(),
    Path(f"{WORK}/splits/train.csv"),
    Path(f"{WORK}/splits/val.csv"),
    Path(f"{WORK}/run10_vs_others.csv"),
]
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        if p.exists():
            z.write(p, p.relative_to(WORK))

print(f"{OUT.name}: {OUT.stat().st_size / 1e6:.1f} MB")
from IPython.display import FileLink, display
display(FileLink(str(OUT)))